# BusLight — Pipeline RAG e avaliação de busca

Notebook com saídas reais da recuperação. A inferência do LLM exige Ollama local; o placar do modelo só é produzido por `python -m src.evaluate_llm`.

In [1]:
from pathlib import Path
import json, sys
sys.path.insert(0, str(Path.cwd()))
from src.knowledge import load_jsonl
from src.retrieval import Retriever
chunks = load_jsonl()
print('Trechos na base:', len(chunks))
print('Exemplo:', chunks[0]['chunk_id'], chunks[0]['source_id'])

Trechos na base: 16
Exemplo: BL-001 BUSLIGHT_INTERNO


In [2]:
from src.evaluate_retrieval import read_cases
cases = read_cases()
print('Casos totais:', len(cases))
print('Perguntas respondíveis:', sum(bool(c['relevant_chunk_ids']) for c in cases))
print('Perguntas para abstenção:', sum(c['should_abstain'] for c in cases))

Casos totais: 21
Perguntas respondíveis: 16
Perguntas para abstenção: 5


In [3]:
retriever = Retriever(chunks)
metrics = retriever.metrics(cases)
print(json.dumps(metrics, ensure_ascii=False, indent=2))

{
  "recall@1": 0.8333333333333334,
  "hit_rate@1": 0.9375,
  "recall@3": 1.0,
  "hit_rate@3": 1.0,
  "answerable_queries": 16,
  "total_relevant_labels": 18
}


In [4]:
for cid in ['Q02','Q06','Q17','Q21']:
    case=next(c for c in cases if c['id']==cid)
    hits=retriever.retrieve(case['question'], top_k=3)
    print(cid, case['question'])
    print([(x['chunk_id'], round(x['score'],4)) for x in hits])

Q02 De onde vêm as posições dos ônibus em tempo real?
[('SPT-002', 0.2997), ('GTFS-001', 0.1078), ('BL-002', 0.104)]
Q06 No GTFS, o que significa um horário 25:35:00?
[('GTFS-002', 0.3286), ('GTFS-001', 0.0864), ('SPT-002', 0.0809)]
Q17 Qual é a lotação exata do ponto da Rua das Flores agora?
[('BL-003', 0.1526), ('BL-002', 0.0983), ('SPT-001', 0.086)]
Q21 Qual é o mascote oficial do BusLight e qual é a cor dele?
[('BL-003', 0.1127), ('BL-002', 0.1012), ('SPT-005', 0.084)]


## Comparação das três configurações

Após instalar/iniciar o Ollama, executar `python -m src.evaluate_llm`. O script roda o mesmo modelo local sem contexto, com RAG e com agente especializado; salva respostas literais e o placar. Não preencher estes resultados manualmente sem executar o modelo.

In [5]:
from pathlib import Path
p=Path('avaliacao/placar_tres_configuracoes.csv')
print(p.read_text(encoding='utf-8-sig') if p.exists() else 'PENDENTE: execute python -m src.evaluate_llm com Ollama local.')

PENDENTE: execute python -m src.evaluate_llm com Ollama local.
